# Orders Benchmarking

This notebook isolates the benchmarking part of the project for `databricks_cat.silver.orders_silver`.

## Goals
* define repeatable benchmark queries
* measure baseline execution time consistently
* inspect physical plans with `EXPLAIN` and `EXPLAIN FORMATTED`
* summarize runtime, table layout, statistics visibility, and pruning hints

## Benchmark queries
* `date_range_filter`
* `product_aggregation`
* `customer_level_aggregation`

## Note on compute
This notebook runs on the current cluster, but for the final project discussion you should note that serverless SQL or Photon-enabled compute is generally preferred for interactive performance workloads.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row
import re
import time

SILVER_TABLE = "databricks_cat.silver.orders_silver"
GOLD_TABLE = "databricks_cat.gold.factorders"

print("Benchmark notebook configured for:")
print(f" - {SILVER_TABLE}")
print(f" - {GOLD_TABLE}")

In [0]:
silver_exists = spark.catalog.tableExists(SILVER_TABLE)
gold_exists = spark.catalog.tableExists(GOLD_TABLE)

if not silver_exists:
    raise ValueError(f"Required table not found: {SILVER_TABLE}")

orders_df = spark.table(SILVER_TABLE)
orders_df.createOrReplaceTempView("orders_silver_v")

factorders_df = spark.table(GOLD_TABLE) if gold_exists else None
if gold_exists:
    factorders_df.createOrReplaceTempView("factorders_v")

availability_df = spark.createDataFrame([
    Row(table_name=SILVER_TABLE, available=silver_exists, role="primary benchmark table"),
    Row(table_name=GOLD_TABLE, available=gold_exists, role="optional downstream context")
])

display(availability_df)
display(orders_df.limit(5))

## Benchmark design

The same three queries are run repeatedly so that later optimization scenarios can be compared fairly.

The date windows are anchored dynamically to the latest `order_date` in the table so the notebook remains reusable after refreshes.

In [0]:
table_detail_cache = {}


def get_table_detail(table_name: str) -> dict:
    if table_name not in table_detail_cache:
        detail_row = spark.sql(f"DESCRIBE DETAIL {table_name}").first().asDict()
        table_detail_cache[table_name] = detail_row
    return table_detail_cache[table_name]


def build_benchmark_queries(table_name: str) -> dict:
    return {
        "date_range_filter": f"""
WITH bounds AS (
    SELECT date_add(max(to_date(order_date)), -90) AS start_date
    FROM {table_name}
)
SELECT *
FROM {table_name}
WHERE to_date(order_date) >= (SELECT start_date FROM bounds)
""",
        "product_aggregation": f"""
WITH bounds AS (
    SELECT date_add(max(to_date(order_date)), -180) AS start_date
    FROM {table_name}
)
SELECT
    product_id,
    SUM(total_amount) AS revenue,
    SUM(quantity) AS units,
    COUNT(*) AS orders
FROM {table_name}
WHERE to_date(order_date) >= (SELECT start_date FROM bounds)
GROUP BY product_id
ORDER BY revenue DESC
""",
        "customer_level_aggregation": f"""
WITH bounds AS (
    SELECT date_add(max(to_date(order_date)), -180) AS start_date
    FROM {table_name}
)
SELECT
    customer_id,
    COUNT(*) AS orders,
    SUM(total_amount) AS revenue,
    AVG(total_amount) AS avg_order_value
FROM {table_name}
WHERE to_date(order_date) >= (SELECT start_date FROM bounds)
GROUP BY customer_id
ORDER BY revenue DESC
"""
    }


STAT_PATTERNS = {
    "full": re.compile(r"statistics:\s*full", re.IGNORECASE),
    "partial": re.compile(r"statistics:\s*partial", re.IGNORECASE),
    "missing": re.compile(r"statistics:\s*missing", re.IGNORECASE)
}


def extract_plan_metrics(query_text: str) -> dict:
    plan_rows = spark.sql(f"EXPLAIN FORMATTED {query_text}").collect()
    plan_text = "\n".join(str(row[0]) for row in plan_rows)

    statistics_line_match = re.search(r"Statistics:\s*([^\n]+)", plan_text, flags=re.IGNORECASE)
    partition_filters_match = re.search(r"PartitionFilters:\s*([^\n]+)", plan_text, flags=re.IGNORECASE)
    pushed_filters_match = re.search(r"PushedFilters:\s*([^\n]+)", plan_text, flags=re.IGNORECASE)
    partition_count_match = re.search(r"PartitionCount:\s*([^\n]+)", plan_text, flags=re.IGNORECASE)
    file_index_match = re.search(r"(?:PreparedDeltaFileIndex|CatalogFileIndex)\(([^\)]*)\)", plan_text, flags=re.IGNORECASE)

    statistics_line = statistics_line_match.group(1).strip() if statistics_line_match else "not reported"

    if STAT_PATTERNS["full"].search(plan_text):
        stats_state = "full"
    elif STAT_PATTERNS["partial"].search(plan_text):
        stats_state = "partial"
    elif STAT_PATTERNS["missing"].search(plan_text):
        stats_state = "missing"
    else:
        stats_state = "not_reported"

    return {
        "statistics_line": statistics_line,
        "stats_state": stats_state,
        "partition_filters": partition_filters_match.group(1).strip() if partition_filters_match else "not reported",
        "pushed_filters": pushed_filters_match.group(1).strip() if pushed_filters_match else "not reported",
        "partition_count": partition_count_match.group(1).strip() if partition_count_match else "not reported",
        "files_scanned_hint": file_index_match.group(1).strip() if file_index_match else "not reported"
    }


def run_benchmarks(table_name: str, scenario_name: str, clear_cached_data: bool = True):
    table_detail = get_table_detail(table_name)
    benchmark_rows = []

    for query_name, query_text in build_benchmark_queries(table_name).items():
        if clear_cached_data:
            spark.catalog.clearCache()

        start_time = time.perf_counter()
        result_df = spark.sql(query_text)
        result_rows = result_df.count()
        elapsed_seconds = round(time.perf_counter() - start_time, 4)

        plan_metrics = extract_plan_metrics(query_text)
        benchmark_rows.append(Row(
            scenario=scenario_name,
            table_name=table_name,
            query_name=query_name,
            elapsed_seconds=elapsed_seconds,
            result_rows=result_rows,
            table_num_files=table_detail.get("numFiles"),
            table_size_bytes=table_detail.get("sizeInBytes"),
            partition_columns=", ".join(table_detail.get("partitionColumns") or []) or "(none)",
            clustering_columns=", ".join(table_detail.get("clusteringColumns") or []) or "(none)",
            statistics_state=plan_metrics["stats_state"],
            statistics_line=plan_metrics["statistics_line"],
            partition_filters=plan_metrics["partition_filters"],
            pushed_filters=plan_metrics["pushed_filters"],
            partitions_read_hint=plan_metrics["partition_count"],
            files_scanned_hint=plan_metrics["files_scanned_hint"]
        ))

    return spark.createDataFrame(benchmark_rows)


print("Benchmark helpers are ready.")

In [0]:
benchmark_queries = build_benchmark_queries(SILVER_TABLE)

for query_name, query_text in benchmark_queries.items():
    print(f"\n--- {query_name} ---")
    print(query_text)

In [0]:
baseline_results_df = run_benchmarks(SILVER_TABLE, "source_external_delta")
baseline_results_df.createOrReplaceTempView("benchmark_results_source_external_delta")
display(baseline_results_df)

In [0]:
%sql
EXPLAIN
WITH bounds AS (
    SELECT date_add(max(to_date(order_date)), -90) AS start_date
    FROM databricks_cat.silver.orders_silver
)
SELECT *
FROM databricks_cat.silver.orders_silver
WHERE to_date(order_date) >= (SELECT start_date FROM bounds)

In [0]:
%sql
EXPLAIN FORMATTED
WITH bounds AS (
    SELECT date_add(max(to_date(order_date)), -180) AS start_date
    FROM databricks_cat.silver.orders_silver
)
SELECT
    product_id,
    SUM(total_amount) AS revenue,
    SUM(quantity) AS units,
    COUNT(*) AS orders
FROM databricks_cat.silver.orders_silver
WHERE to_date(order_date) >= (SELECT start_date FROM bounds)
GROUP BY product_id
ORDER BY revenue DESC

In [0]:
%sql
EXPLAIN FORMATTED
WITH bounds AS (
    SELECT date_add(max(to_date(order_date)), -180) AS start_date
    FROM databricks_cat.silver.orders_silver
)
SELECT
    customer_id,
    COUNT(*) AS orders,
    SUM(total_amount) AS revenue,
    AVG(total_amount) AS avg_order_value
FROM databricks_cat.silver.orders_silver
WHERE to_date(order_date) >= (SELECT start_date FROM bounds)
GROUP BY customer_id
ORDER BY revenue DESC

In [0]:
summary_df = (
    baseline_results_df
    .select(
        "query_name",
        "elapsed_seconds",
        "result_rows",
        "table_num_files",
        "table_size_bytes",
        "partition_columns",
        "clustering_columns",
        "statistics_state",
        "statistics_line",
        "partition_filters",
        "pushed_filters",
        "partitions_read_hint",
        "files_scanned_hint"
    )
    .orderBy("elapsed_seconds")
)

display(summary_df)

slowest_row = baseline_results_df.orderBy(F.desc("elapsed_seconds")).first()
fastest_row = baseline_results_df.orderBy(F.asc("elapsed_seconds")).first()

print("Benchmark findings:")
print(f"- Fastest query: {fastest_row['query_name']} at {fastest_row['elapsed_seconds']:.4f} seconds")
print(f"- Slowest query: {slowest_row['query_name']} at {slowest_row['elapsed_seconds']:.4f} seconds")
print("- Source table is external Delta, so later optimization tests can compare managed layout and clustering improvements.")